In [344]:
!pip install -q opencv-python

In [345]:
pip show opencv-python

Name: opencv-python
Version: 4.13.0.92
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: C:\Users\almaa\AppData\Roaming\Python\Python313\site-packages
Requires: numpy
Required-by: supervision, ultralytics
Note: you may need to restart the kernel to use updated packages.


In [346]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request

In [347]:
print("OpenCV version:", cv2.__version__)
print("OpenCV location:", cv2.__file__)
print("CascadeClassifier available:",
      hasattr(cv2, "CascadeClassifier"))

OpenCV version: 4.10.0
OpenCV location: C:\Users\almaa\AppData\Roaming\Python\Python313\site-packages\cv2\__init__.py
CascadeClassifier available: True


In [348]:
MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory:", os.path.abspath(MODEL_DIR))

Model directory: c:\Users\almaa\OneDrive\Desktop\additionalProject\models


In [349]:
MODEL_URL = (
    "https://storage.openvinotoolkit.org/"
    "repositories/open_model_zoo/public/2022.1/"
    "open-closed-eye-0001/open_closed_eye.onnx"
)

MODEL_PATH = os.path.join("models", "eye_classifier.onnx")

if not os.path.exists(MODEL_PATH):
    print("Downloading eye classifier...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download complete.")
else:
    print("Model already exists.")

print(
    "Model size:",
    round(os.path.getsize(MODEL_PATH) / 1024, 2),
    "KB"
)

Model already exists.
Model size: 16977.33 KB


In [350]:
model_size = os.path.getsize(MODEL_PATH)

print(
    "Model size:",
    round(model_size / (1024 * 1024), 2),
    "MB"
)

Model size: 16.58 MB


In [351]:
eye_net = cv2.dnn.readNetFromONNX(MODEL_PATH)

print("Eye classifier loaded successfully.")

Eye classifier loaded successfully.


In [352]:
print("Model loaded:", eye_net is not None)

Model loaded: True


In [353]:
print(cv2.data.haarcascades)

C:\Users\almaa\AppData\Roaming\Python\Python313\site-packages\cv2\data\


In [354]:
FACE_CASCADE_PATH = (
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

EYE_CASCADE_PATH = (
    cv2.data.haarcascades +
    "haarcascade_eye.xml"
)

In [355]:
face_cascade = cv2.CascadeClassifier(
    FACE_CASCADE_PATH
)

eye_cascade = cv2.CascadeClassifier(
    EYE_CASCADE_PATH
)

In [356]:
print(
    "Face cascade loaded:",
    not face_cascade.empty()
)

print(
    "Eye cascade loaded:",
    not eye_cascade.empty()
)

Face cascade loaded: True
Eye cascade loaded: True


In [357]:
IMAGE_PATH = "test2.jpg"

image = cv2.imread(IMAGE_PATH)

if image is None:
    raise FileNotFoundError(
        f"Could not read image: {IMAGE_PATH}"
    )

print("Image shape:", image.shape)

Image shape: (408, 612, 3)


In [358]:
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

In [359]:
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(80, 80)
)

print("Faces detected:", len(faces))
print(faces)

Faces detected: 1
[[209  81 194 194]]


In [360]:
display_image = image.copy()

for (x, y, w, h) in faces:

    cv2.rectangle(
        display_image,
        (x, y),
        (x + w, y + h),
        (255, 0, 0),
        2
    )

plt.figure(figsize=(10, 7))

plt.imshow(
    cv2.cvtColor(
        display_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [361]:
def preprocess_eye(eye_bgr):

    eye = cv2.resize(
        eye_bgr,
        (32, 32),
        interpolation=cv2.INTER_AREA
    )

    blob = cv2.dnn.blobFromImage(
        eye,
        scalefactor=1.0 / 255.0,
        size=(32, 32),
        mean=(0, 0, 0),
        swapRB=False,
        crop=False
    )

    return blob.astype(np.float32)

In [362]:
def softmax(x):
    """
    Convert logits into probabilities.
    """

    x = x - np.max(x)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x)

In [363]:
def classify_eye(eye_bgr):

    input_blob = preprocess_eye(eye_bgr)

    eye_net.setInput(input_blob)

    output = eye_net.forward()

    probabilities = output.flatten()

    class_id = int(np.argmax(probabilities))

    confidence = float(probabilities[class_id])

    # IMPORTANT:
    # OpenVINO model:
    # 0 = Open
    # 1 = Closed

    if class_id == 0:
        label = "Open"
    else:
        label = "Closed"

    return label, confidence, probabilities

In [364]:
eye_image = cv2.imread(
    "test2.jpg"
)

if eye_image is None:
    raise FileNotFoundError(
        "Eye image could not be loaded."
    )

In [365]:
label, confidence, probabilities = classify_eye(
    eye_image
)

print("Prediction:", label)
print(
    "Confidence:",
    f"{confidence * 100:.2f}%"
)

print(
    "Probabilities:",
    probabilities
)

Prediction: Closed
Confidence: 6.63%
Probabilities: [-0.25761843  0.06628953]


In [379]:
# plt.figure(figsize=(5, 3))

# plt.imshow(
#     cv2.cvtColor(
#         eye_image,
#         cv2.COLOR_BGR2RGB
#     )
# )

# plt.title(
#     f"{label} "
# )

# plt.axis("off")
# plt.show()

In [ ]:
def detect_eyes(face_bgr):
    """
    Detect eyes inside a face crop.

    Returns:
        list of tuples:
        (x, y, w, h, eye_crop)
    """

    gray_face = cv2.cvtColor(
        face_bgr,
        cv2.COLOR_BGR2GRAY
    )

    h, w = gray_face.shape

    upper_face = gray_face[
        0:int(h * 0.65),
        :
    ]

    eyes = eye_cascade.detectMultiScale(
        upper_face,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(20, 20)
    )

    results = []

    for (x, y, ew, eh) in eyes:

        eye_crop = face_bgr[
            y:y+eh,
            x:x+ew
        ]

        if eye_crop.size == 0:
            continue

        results.append(
            (
                x,
                y,
                ew,
                eh,
                eye_crop
            )
        )

    results.sort(
        key=lambda item: item[0]
    )

    return results[:2]

In [ ]:
def process_image(image):
    """
    Complete image pipeline:

    Image
       ↓
    Face detection
       ↓
    Face crop
       ↓
    Eye detection
       ↓
    Eye crop
       ↓
    MobileNetV2
       ↓
    Open / Closed
    """

    output = image.copy()

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )

    all_results = []

    for (fx, fy, fw, fh) in faces:

        # Face crop
        face_crop = image[
            fy:fy+fh,
            fx:fx+fw
        ]

        # Draw face
        cv2.rectangle(
            output,
            (fx, fy),
            (fx + fw, fy + fh),
            (255, 0, 0),
            2
        )


        eyes = detect_eyes(
            face_crop
        )

        for (
            ex,
            ey,
            ew,
            eh,
            eye_crop
        ) in eyes:

            label, confidence, probabilities = classify_eye(
                eye_crop
            )

            x1 = fx + ex
            y1 = fy + ey

            x2 = x1 + ew
            y2 = y1 + eh

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            text = (
                f"{label} "
                
            )

            cv2.putText(
                output,
                text,
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )

            all_results.append({
                "face_bbox": (
                    fx,
                    fy,
                    fw,
                    fh
                ),
                "eye_bbox": (
                    x1,
                    y1,
                    ew,
                    eh
                ),
                "label": label,
                "confidence": confidence,
                "eye_crop": eye_crop
            })

    return output, all_results

In [369]:
image = cv2.imread(
    "test2.jpg"
)

if image is None:
    raise FileNotFoundError(
        "Could not load test image."
    )

result_image, results = process_image(
    image
)

In [370]:
for i, result in enumerate(results):

    print(
        f"Eye {i+1}: "
        f"{result['label']} "
        f"({result['confidence']*100:.2f}%)"
    )

Eye 1: Open (10.86%)
Eye 2: Open (18.57%)


In [371]:
plt.figure(figsize=(12, 8))

plt.imshow(
    cv2.cvtColor(
        result_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [372]:
for i, result in enumerate(results):

    eye_crop = result["eye_crop"]

    plt.figure(figsize=(4, 2))

    plt.imshow(
        cv2.cvtColor(
            eye_crop,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"Eye {i+1}: "
        f"{result['label']} "
        f"({result['confidence']*100:.1f}%)"
    )

    plt.axis("off")
    plt.show()

In [373]:
def detect_eye_state(image_path):

    image = cv2.imread(
        image_path
    )

    if image is None:
        raise FileNotFoundError(
            f"Cannot read: {image_path}"
        )

    result_image, results = process_image(
        image
    )

    return result_image, results

In [374]:
result_image, results = detect_eye_state(
    "test2.jpg"
)

In [375]:
eye = results[0]["eye_crop"]

plt.figure(figsize=(5, 3))
plt.imshow(cv2.cvtColor(eye, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [376]:
tensor = preprocess_eye(eye)

print("Tensor shape:", tensor.shape)
print("Tensor dtype:", tensor.dtype)
print("Tensor min:", tensor.min())
print("Tensor max:", tensor.max())

Tensor shape: (1, 3, 32, 32)
Tensor dtype: float32
Tensor min: 0.019607844
Tensor max: 0.96862745


In [377]:
eye_net.setInput(tensor)

output = eye_net.forward()

print("Raw output:", output)
print("Output shape:", output.shape)

Raw output: [[0.10856023 0.09203999]]
Output shape: (1, 2)


In [378]:
probabilities = softmax(output.flatten())

print("Class 0 - Closed:", probabilities[0])
print("Class 1 - Open:", probabilities[1])

predicted_class = np.argmax(probabilities)

print("Predicted class:", predicted_class)

Class 0 - Closed: 0.50412995
Class 1 - Open: 0.49587005
Predicted class: 0
